# Membuat Edges (Hubungan Guru dan Murid)

## Import Library

In [2]:
import pandas as pd
import ast
import re

## Load Data

In [3]:
# pd.set_option('display.max_colwidth', None)

df = pd.read_csv('../data/processed/sanadset_cleaned.csv', header=None)
df.columns = ['Hadith', 'Book', 'Num_hadith', 'Matn', 'Sanad', 'Sanad_Length', 'Sanad_no_harakat']
df = df.iloc[1:].reset_index(drop=True)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13016\1150346939.py:3: DtypeWarning: Columns (0: 2, 1: 5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/processed/sanadset_cleaned.csv', header=None)


### Menghitung data dan Memecah isi list pada kolom Sanad_no_harakat

In [4]:
df['Sanad_no_harakat'] = df['Sanad_no_harakat'].apply(ast.literal_eval)


In [5]:
df.shape

(487451, 7)

In [6]:
print(type(df['Sanad_no_harakat'].iloc[0]))

<class 'list'>


In [7]:
sanad_exploded = df.explode('Sanad_no_harakat')
display(sanad_exploded.head())


,Hadith,Book,Num_hadith,Matn,Sanad,Sanad_Length,Sanad_no_harakat
0,حَدَّثَنِي <SANAD> <NAR> أَبُو عُبَيْدَةَ مُسْ...,مسند الربيع بن حبيب,1,، عَنِ النَّبِيِّ صَلَّى اللَّهُ عَلَيْهِ وَ...,['أَبُو عُبَيْدَةَ مُسْلِمُ بْنُ أَبِي كَرِيمَ...,3,ابو عبيده مسلم بن ابو كريمه التميمي
0,حَدَّثَنِي <SANAD> <NAR> أَبُو عُبَيْدَةَ مُسْ...,مسند الربيع بن حبيب,1,، عَنِ النَّبِيِّ صَلَّى اللَّهُ عَلَيْهِ وَ...,['أَبُو عُبَيْدَةَ مُسْلِمُ بْنُ أَبِي كَرِيمَ...,3,جابر بن زيد الازدي
0,حَدَّثَنِي <SANAD> <NAR> أَبُو عُبَيْدَةَ مُسْ...,مسند الربيع بن حبيب,1,، عَنِ النَّبِيِّ صَلَّى اللَّهُ عَلَيْهِ وَ...,['أَبُو عُبَيْدَةَ مُسْلِمُ بْنُ أَبِي كَرِيمَ...,3,عبد الله بن عباس
1,حَدَّثَنِي <SANAD> <NAR> أَبُو عُبَيْدَةَ </NA...,مسند الربيع بن حبيب,3,، أُمِّ الْمُؤْمِنِينَ ، رَضِيَ اللَّهُ عَنْ...,"['أَبُو عُبَيْدَةَ', 'جَابِرِ بْنِ زَيْدٍ', 'ع...",3,ابو عبيده
1,حَدَّثَنِي <SANAD> <NAR> أَبُو عُبَيْدَةَ </NA...,مسند الربيع بن حبيب,3,، أُمِّ الْمُؤْمِنِينَ ، رَضِيَ اللَّهُ عَنْ...,"['أَبُو عُبَيْدَةَ', 'جَابِرِ بْنِ زَيْدٍ', 'ع...",3,جابر بن زيد


In [8]:
sanad_exploded.shape

(2993196, 7)

In [9]:
df.Sanad_no_harakat.head(10)

0    [ابو عبيده مسلم بن ابو كريمه التميمي, جابر بن ...
1                      [ابو عبيده, جابر بن زيد, عائشه]
2                             [ابو عبيده, جابر بن زيد]
3                  [ابو عبيده, جابر بن زيد, ابو هريره]
4            [ابو عبيده, جابر بن زيد, ابو سعيد الخدري]
5                   [ابو عبيده, جابر بن زيد, ابن عباس]
6                [ابو عبيده, جابر بن زيد, انس بن مالك]
7            [ابو عبيده, جابر بن زيد, ابو سعيد الخدري]
8                  [ابو عبيده, جابر بن زيد, ابو هريره]
9              [ابو عبيده, جابر بن زيد, عمر بن الخطاب]
Name: Sanad_no_harakat, dtype: object

### Menghitung Perawi (Narrator) Paling Banyak Muncul

In [10]:
from collections import Counter

# Fungsi normalisasi nama Arab
def normalize_arabic_name(text):
    text = str(text)

    # Hilangkan spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    # Hapus akhiran " L" jika ada
    text = re.sub(r'\s+L$', '', text)

    # Normalisasi huruf Arab
    text = text.replace('أ', 'ا')
    text = text.replace('إ', 'ا')
    text = text.replace('آ', 'ا')

    text = text.replace('ى', 'ي')
    text = text.replace('ة', 'ه')

    return text

# Flatten semua nama perawi dari list Sanad_no_harakat
all_narrators = []

for sanad_list in df['Sanad_no_harakat']:
    if isinstance(sanad_list, list):
        all_narrators.extend(sanad_list)

# Normalisasi nama perawi
all_narrators_normalized = [
    normalize_arabic_name(name)
    for name in all_narrators
]

# Hitung frekuensi setelah normalisasi
narrator_counts = Counter(all_narrators_normalized)

# DataFrame hasil
narrator_df = pd.DataFrame(
    narrator_counts.most_common(),
    columns=['Perawi', 'Jumlah']
)

# Tambahkan kolom Perawi_ID
narrator_df.insert(
    0,
    'Perawi_ID',
    range(1, len(narrator_df) + 1)
)


display(narrator_df)

,Perawi_ID,Perawi,Jumlah
0,1,ابو هريره,51669
1,2,ابن عباس,48300
2,3,عائشه,33145
3,4,ابن عمر,28892
4,5,شعبه,26570
...,...,...,...
177284,177285,جد بن فضيل,1
177285,177286,ابو سلمه الخرساني,1
177286,177287,الاشعث بن طلق,1
177287,177288,حفص بن موسي,1


In [11]:
mapping = []

for sanad_list in df['Sanad_no_harakat']:
    if isinstance(sanad_list, list):
        for name in sanad_list:
            mapping.append({
                'Nama_Asli': name,
                'Nama_Normalisasi': normalize_arabic_name(name)
            })

mapping_df = pd.DataFrame(mapping)

merged = (
    mapping_df
    .groupby('Nama_Normalisasi')['Nama_Asli']
    .unique()
    .reset_index()
)

merged['Jumlah_Varian'] = merged['Nama_Asli'].apply(len)

merged = merged[
    merged['Jumlah_Varian'] > 1
].sort_values(
    'Jumlah_Varian',
    ascending=False
)

display(merged)

,Nama_Normalisasi,Nama_Asli,Jumlah_Varian


### Buat Hubungan Guru-Murid dari Sanad


In [12]:
from IPython.display import display

# Fungsi membentuk hubungan guru-murid
def build_teacher_student_edges(chain):
    if not isinstance(chain, list) or len(chain) < 2:
        return []
    
    # Format: (Guru, Murid)
    return [(chain[i + 1], chain[i]) for i in range(len(chain) - 1)]

# Pastikan semua data berupa list
def ensure_list(cell):
    if isinstance(cell, list):
        return cell
    
    if isinstance(cell, str) and cell.strip().startswith('['):
        try:
            return ast.literal_eval(cell)
        except Exception:
            return [cell]
    
    if pd.isna(cell):
        return []
    
    return [cell]

# Ubah kolom menjadi list
sanad_lists = df['Sanad_no_harakat'].apply(ensure_list)

# Membentuk semua hubungan guru-murid
edges = []
for sanad in sanad_lists:
    edges.extend(build_teacher_student_edges(sanad))

# Membuat DataFrame hubungan
edges_df = pd.DataFrame(edges, columns=['Guru', 'Murid'])

# Hitung weight tanpa mengubah urutan awal
edges_df['Weight'] = (
    edges_df
    .groupby(['Guru', 'Murid'])['Guru']
    .transform('count')
)

# Hapus duplikat tetapi tetap menjaga urutan awal
edges_df = edges_df.drop_duplicates(subset=['Guru', 'Murid'])

# Reset index
edges_df = edges_df.reset_index(drop=True)

# Output
display(edges_df[['Guru', 'Murid', 'Weight']])

,Guru,Murid,Weight
0,جابر بن زيد الازدي,ابو عبيده مسلم بن ابو كريمه التميمي,1
1,عبد الله بن عباس,جابر بن زيد الازدي,1
2,جابر بن زيد,ابو عبيده,522
3,عائشه,جابر بن زيد,68
4,ابو هريره,جابر بن زيد,76
...,...,...,...
781056,ابي مسعود البدري,حذيفه بن اليمان,1
781057,سعيد بن العاص,ابي مسعود البدري,1
781058,ابو مسعود,سعيد بن العاص,1
781059,عبد الله,علي بن علقمه,1


In [13]:
type(df['Sanad_no_harakat'].iloc[0])

list

In [14]:
type(edges_df['Guru'].iloc[0])

str

### Menghapus Lafal Periwayatan

In [15]:
# Normalisasi terakhir
def clean_final_name(s):
    if pd.isna(s):
        return ""

    s = str(s)

    s = re.sub(r'\s+', ' ', s).strip()
    s = re.sub(r'\s+L$', '', s)


    # Hapus karakter tersembunyi
    s = re.sub(r'[\u200b-\u200f\u202a-\u202e\u2060\ufeff]', '', s)

    # Sisakan huruf Arab dan spasi
    s = re.sub(r'[^\u0600-\u06FF\s]', ' ', s)

    # Rapikan spasi
    s = re.sub(r'\s+', ' ', s).strip()

    return s


# Cleaning nama
edges_df['Guru'] = edges_df['Guru'].apply(clean_final_name)
edges_df['Murid'] = edges_df['Murid'].apply(clean_final_name)

# Hapus relasi tidak valid
edges_df = edges_df[
    (edges_df['Guru'].str.len() > 0) &
    (edges_df['Murid'].str.len() > 0) &
    (edges_df['Guru'] != edges_df['Murid'])
].copy()

# Gabungkan relasi sama
edges_df = (
    edges_df
    .groupby(['Guru', 'Murid'], as_index=False, sort=False)
    .agg(Weight=('Weight', 'sum'))
)

# Inverse weight
edges_df['Inverse_Weight'] = edges_df['Weight'].apply(
    lambda x: 1 / x if x > 0 else 0
)

# Reset index
edges_df = edges_df.reset_index(drop=True)

display(edges_df)

,Guru,Murid,Weight,Inverse_Weight
0,جابر بن زيد الازدي,ابو عبيده مسلم بن ابو كريمه التميمي,1,1.000000
1,عبد الله بن عباس,جابر بن زيد الازدي,1,1.000000
2,جابر بن زيد,ابو عبيده,522,0.001916
3,عائشه,جابر بن زيد,68,0.014706
4,ابو هريره,جابر بن زيد,76,0.013158
...,...,...,...,...
776793,ابي مسعود البدري,حذيفه بن اليمان,1,1.000000
776794,سعيد بن العاص,ابي مسعود البدري,1,1.000000
776795,ابو مسعود,سعيد بن العاص,1,1.000000
776796,عبد الله,علي بن علقمه,1,1.000000


### Mengecek karakter selain huruf arab

In [16]:
# Cek karakter selain huruf Arab
def check_non_arabic(text):

    if not isinstance(text, str):
        return False

    # Cari karakter selain Arab dan spasi
    return bool(
        re.search(r'[^\u0600-\u06FF\s]', text)
    )

# Ambil data yang masih memiliki
# karakter non-Arab pada kolom Guru atau Murid
non_arabic_df = edges_df[
    edges_df['Guru'].apply(check_non_arabic) |
    edges_df['Murid'].apply(check_non_arabic)
]

# Tampilkan hasil
display(non_arabic_df[['Guru', 'Murid']].head(20))

# Jumlah data
print("Jumlah data dengan karakter non-Arab:")
print(len(non_arabic_df))

,Guru,Murid


Jumlah data dengan karakter non-Arab:
0


### Menghapus karakter selain huruf arab

In [17]:
# Fungsi untuk menghapus karakter selain Arab dan spasi
def remove_non_arabic(text):
    if not isinstance(text, str):
        return text

    # Hapus karakter selain Arab dan spasi
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)

    # Rapikan spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Terapkan ke kolom Guru dan Murid
edges_df['Guru'] = edges_df['Guru'].apply(remove_non_arabic)
edges_df['Murid'] = edges_df['Murid'].apply(remove_non_arabic)

# Cek ulang apakah masih ada karakter non-Arab
non_arabic_df = edges_df[
    edges_df['Guru'].apply(check_non_arabic) |
    edges_df['Murid'].apply(check_non_arabic)
]

print("Jumlah data dengan karakter non-Arab setelah dibersihkan:")
print(len(non_arabic_df))

display(edges_df[['Guru', 'Murid']])

Jumlah data dengan karakter non-Arab setelah dibersihkan:
0


,Guru,Murid
0,جابر بن زيد الازدي,ابو عبيده مسلم بن ابو كريمه التميمي
1,عبد الله بن عباس,جابر بن زيد الازدي
2,جابر بن زيد,ابو عبيده
3,عائشه,جابر بن زيد
4,ابو هريره,جابر بن زيد
...,...,...
776793,ابي مسعود البدري,حذيفه بن اليمان
776794,سعيد بن العاص,ابي مسعود البدري
776795,ابو مسعود,سعيد بن العاص
776796,عبد الله,علي بن علقمه


In [18]:
edges_df = edges_df.dropna(subset=['Guru', 'Murid'])

### remove text

In [19]:
remove_text = "أن أم الفضل بنت الحارث بعثته إلى معاوية بالشام"

edges_df = edges_df[
    ~edges_df['Guru'].str.contains(remove_text, na=False) &
    ~edges_df['Murid'].str.contains(remove_text, na=False)
]
edges_df = edges_df.reset_index(drop=True)
edges_df.shape

(776798, 4)

### Membersihkan data hubungan Guru dan Murid jika mengandung angka

In [20]:
edges_df = edges_df[
    ~edges_df['Guru'].str.contains(r'\d', na=False) &
    ~edges_df['Murid'].str.contains(r'\d', na=False)
]
edges_df = edges_df.reset_index(drop=True)
edges_df.shape 

(776798, 4)

In [21]:
# Membersihkan data relasi guru-murid supaya tidak ada relasi yang salah karena salah satu namanya kosong.
invalid_relations = edges_df[
    (edges_df['Guru'] == '') |
    (edges_df['Murid'] == '')
]

print("Jumlah relasi tidak valid (Guru atau Murid kosong):", len(invalid_relations))

display(invalid_relations[['Guru', 'Murid', 'Weight']])

edges_df = edges_df[
    (edges_df['Guru'] != '') &
    (edges_df['Murid'] != '')
]

edges_df = edges_df.reset_index(drop=True)
display(edges_df[['Guru', 'Murid', 'Weight']])
edges_df.shape

Jumlah relasi tidak valid (Guru atau Murid kosong): 0


,Guru,Murid,Weight


,Guru,Murid,Weight
0,جابر بن زيد الازدي,ابو عبيده مسلم بن ابو كريمه التميمي,1
1,عبد الله بن عباس,جابر بن زيد الازدي,1
2,جابر بن زيد,ابو عبيده,522
3,عائشه,جابر بن زيد,68
4,ابو هريره,جابر بن زيد,76
...,...,...,...
776793,ابي مسعود البدري,حذيفه بن اليمان,1
776794,سعيد بن العاص,ابي مسعود البدري,1
776795,ابو مسعود,سعيد بن العاص,1
776796,عبد الله,علي بن علقمه,1


(776798, 4)

In [22]:
# Hilangkan spasi berlebih terlebih dahulu
edges_df['Guru'] = edges_df['Guru'].astype(str).str.strip()
edges_df['Murid'] = edges_df['Murid'].astype(str).str.strip()

# Cari relasi yang sama
same_relation = edges_df[
    edges_df['Guru'] == edges_df['Murid']
]

print('Jumlah relasi guru-murid yang sama:', len(same_relation))

display(same_relation[['Guru', 'Murid', 'Weight']])

# Hapus relasi yang sama
edges_df = edges_df[
    edges_df['Guru'] != edges_df['Murid']
]

# Reset index
edges_df = edges_df.reset_index(drop=True)

print('Jumlah data setelah relasi sama dihapus:', len(edges_df))

display(edges_df[['Guru', 'Murid', 'Weight']])

edges_df.shape
# edges_df = edges_df[
#     edges_df['Guru'] != edges_df['Murid']
# ]
# edges_df = edges_df.reset_index(drop=True)
# edges_df.shape

# display(edges_df[['Guru', 'Murid', 'Weight']])

Jumlah relasi guru-murid yang sama: 0


,Guru,Murid,Weight


Jumlah data setelah relasi sama dihapus: 776798


,Guru,Murid,Weight
0,جابر بن زيد الازدي,ابو عبيده مسلم بن ابو كريمه التميمي,1
1,عبد الله بن عباس,جابر بن زيد الازدي,1
2,جابر بن زيد,ابو عبيده,522
3,عائشه,جابر بن زيد,68
4,ابو هريره,جابر بن زيد,76
...,...,...,...
776793,ابي مسعود البدري,حذيفه بن اليمان,1
776794,سعيد بن العاص,ابي مسعود البدري,1
776795,ابو مسعود,سعيد بن العاص,1
776796,عبد الله,علي بن علقمه,1


(776798, 4)

In [23]:
# Daftar lafaz periwayatan
hapus_kata = [
    "ثنا", "حدثنا", "قال حدثنا", "وقال", "عن",
    "أخبرنا", "أنبأنا", "حدثني", "حدثه", "سمعت",
    "يقول", "ذكر", "رواه", "حدث", "حكى"
]



# Regex
pola = r'\b(?:' + '|'.join(map(re.escape, hapus_kata)) + r')\b'

mengandung_lafaz = edges_df[
    edges_df['Guru'].str.contains(pola, na=False) |
    edges_df['Murid'].str.contains(pola, na=False)
]

display(mengandung_lafaz)
# Hapus lafaz dari kolom Guru dan Murid
edges_df['Guru'] = (
    edges_df['Guru']
    .str.replace(pola, '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

edges_df['Murid'] = (
    edges_df['Murid']
    .str.replace(pola, '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# Tampilkan hasil
display(edges_df)

,Guru,Murid,Weight,Inverse_Weight


,Guru,Murid,Weight,Inverse_Weight
0,جابر بن زيد الازدي,ابو عبيده مسلم بن ابو كريمه التميمي,1,1.000000
1,عبد الله بن عباس,جابر بن زيد الازدي,1,1.000000
2,جابر بن زيد,ابو عبيده,522,0.001916
3,عائشه,جابر بن زيد,68,0.014706
4,ابو هريره,جابر بن زيد,76,0.013158
...,...,...,...,...
776793,ابي مسعود البدري,حذيفه بن اليمان,1,1.000000
776794,سعيد بن العاص,ابي مسعود البدري,1,1.000000
776795,ابو مسعود,سعيد بن العاص,1,1.000000
776796,عبد الله,علي بن علقمه,1,1.000000


In [24]:
# Menampilkan data duplikat beserta karakter tersembunyi

# Ambil semua relasi duplikat
duplikat_df = edges_df[
    edges_df.duplicated(
        subset=['Guru', 'Murid'],
        keep=False
    )
]

# Urutkan agar mudah dibaca
duplikat_df = duplikat_df.sort_values(
    by=['Guru', 'Murid']
).reset_index(drop=True)

# Tampilkan dataframe biasa
display(duplikat_df)


# Menampilkan relasi guru-murid yang duplikat

duplikat_df = edges_df[
    edges_df.duplicated(
        subset=['Guru', 'Murid'],
        keep=False
    )
]

for i, row in duplikat_df.iterrows():
    print(repr(row['Guru']), repr(row['Murid']))

# Urutkan agar mudah dilihat
duplikat_df = duplikat_df.sort_values(
    by=['Guru', 'Murid']
).reset_index(drop=True)

display(duplikat_df)

# Hapus duplikat tetapi tetap menjaga urutan awal
edges_df = edges_df.drop_duplicates(subset=['Guru', 'Murid'])

# Reset index
edges_df = edges_df.reset_index(drop=True)

# Output
display(edges_df[['Guru', 'Murid', 'Weight', 'Inverse_Weight']])

,Guru,Murid,Weight,Inverse_Weight


,Guru,Murid,Weight,Inverse_Weight


,Guru,Murid,Weight,Inverse_Weight
0,جابر بن زيد الازدي,ابو عبيده مسلم بن ابو كريمه التميمي,1,1.000000
1,عبد الله بن عباس,جابر بن زيد الازدي,1,1.000000
2,جابر بن زيد,ابو عبيده,522,0.001916
3,عائشه,جابر بن زيد,68,0.014706
4,ابو هريره,جابر بن زيد,76,0.013158
...,...,...,...,...
776793,ابي مسعود البدري,حذيفه بن اليمان,1,1.000000
776794,سعيد بن العاص,ابي مسعود البدري,1,1.000000
776795,ابو مسعود,سعيد بن العاص,1,1.000000
776796,عبد الله,علي بن علقمه,1,1.000000


In [25]:
def clean_text(s):
    if pd.isna(s):
        return s
    
    # hapus karakter non-printable TAPI jangan hapus huruf Arab
    s = re.sub(r"[^\x20-\x7E\u0600-\u06FF\s]", "", str(s))
    # rapikan spasi
    return re.sub(r"\s+", " ", s).strip()

edges_df['Guru'] = edges_df['Guru'].apply(clean_text)
edges_df['Murid'] = edges_df['Murid'].apply(clean_text)

# Reset index
edges_df = edges_df.reset_index(drop=True)

# Output
display(edges_df[['Guru', 'Murid', 'Weight', 'Inverse_Weight']])

,Guru,Murid,Weight,Inverse_Weight
0,جابر بن زيد الازدي,ابو عبيده مسلم بن ابو كريمه التميمي,1,1.000000
1,عبد الله بن عباس,جابر بن زيد الازدي,1,1.000000
2,جابر بن زيد,ابو عبيده,522,0.001916
3,عائشه,جابر بن زيد,68,0.014706
4,ابو هريره,جابر بن زيد,76,0.013158
...,...,...,...,...
776793,ابي مسعود البدري,حذيفه بن اليمان,1,1.000000
776794,سعيد بن العاص,ابي مسعود البدري,1,1.000000
776795,ابو مسعود,سعيد بن العاص,1,1.000000
776796,عبد الله,علي بن علقمه,1,1.000000


In [26]:
print(edges_df.dtypes)

Guru                  str
Murid                 str
Weight              int64
Inverse_Weight    float64
dtype: object


## Transliterasi Nama Guru dan Murid: Arab → Indonesia

In [27]:
NAMA_KAMUS = {
    'ابو': 'Abu', 'ابي': 'Abi', 'ابن': 'Ibnu', 'بن': 'bin',
    'بنت': 'binti', 'ام': 'Umm', 'عبد': 'Abd',
    'الله': 'Allah', 'عبد الله': 'Abdullah',

    'ابو هريره': 'Abu Hurairah',   'ابو هريرة': 'Abu Hurairah',
    'ابن عباس': 'Ibnu Abbas',
    'عائشه': 'Aisyah',             'عائشة': 'Aisyah',
    'الزهري': 'Az-Zuhri',
    'شعبه': "Syu'bah",             'شعبة': "Syu'bah",
    'سفيان': 'Sufyan',
    'قتاده': 'Qatadah',            'قتادة': 'Qatadah',
    'نافع': "Nafi'",
    'ابن عمر': 'Ibnu Umar',
    'مالك': 'Malik',
    'معمر': "Ma'mar",
    'انس': 'Anas',                 'أنس': 'Anas',
    'انس بن مالك': 'Anas bin Malik',
    'ابراهيم': 'Ibrahim',          'إبراهيم': 'Ibrahim',
    'الاعمش': "Al-A'masy",
    'ابو صالح': 'Abu Shalih',
    'ابو سلمه': 'Abu Salamah',     'ابو سلمة': 'Abu Salamah',
    'عكرمه': 'Ikrimah',            'عكرمة': 'Ikrimah',
    'عروه': 'Urwah',               'عروة': 'Urwah',
    'الاعرج': "Al-A'raj",
    'ابو الزبير': 'Abu Az-Zubair',
    'ابو اسحاق': 'Abu Ishaq',
    'عبد الرزاق': 'Abdurrazzaq',
    'حماد بن سلمه': 'Hammad bin Salamah',
    'ابو بشر': 'Abu Bisyr',
    'ثابت': 'Tsabit',
    'ابو معاويه': "Abu Mu'awiyah", 'ابو معاوية': "Abu Mu'awiyah",
    'سعيد بن جبير': "Sa'id bin Jubair",
    'هشام بن عروه': 'Hisyam bin Urwah',
    'ابن شهاب': 'Ibnu Syihab',
    'ابن جريج': 'Ibnu Juraij',
    'محمد بن جعفر': "Muhammad bin Ja'far",
    'ابو هشام بن عروه': 'Abu Hisyam bin Urwah',
    'ابو العباس': 'Abul Abbas',
    'ابو عبد الله': 'Abu Abdillah',

    'هريره': 'Hurairah',   'هريرة': 'Hurairah',
    'عباس': 'Abbas',       'صالح': 'Shalih',
    'سلمه': 'Salamah',     'سلمة': 'Salamah',
    'اعمش': "A'masy",      'زهري': 'Zuhri',
    'رزاق': 'Razzaq',      'الرزاق': 'Ar-Razzaq',
    'حماد': 'Hammad',      'بشر': 'Bisyr',
    'معاويه': "Mu'awiyah", 'معاوية': "Mu'awiyah",
    'سعيد': "Sa'id",       'جبير': 'Jubair',
    'هشام': 'Hisyam',      'شهاب': 'Syihab',
    'جريج': 'Juraij',      'جعفر': "Ja'far",
    'محمد': 'Muhammad',    'احمد': 'Ahmad',
    'أحمد': 'Ahmad',       'حنبل': 'Hanbal',
    'يعقوب': "Ya'qub",     'الحافظ': 'Al-Hafizh',
    'حفص': 'Hafs',         'حسن': 'Hasan',
    'حسين': 'Husain',      'علي': 'Ali',
    'عثمان': 'Utsman',     'بكر': 'Bakar',
    'خالد': 'Khalid',
    'حمزه': 'Hamzah',      'حمزة': 'Hamzah',
    'يحيي': 'Yahya',       'يحيى': 'Yahya',
    'موسي': 'Musa',        'موسى': 'Musa',
    'عيسي': 'Isa',         'عيسى': 'Isa',
    'داود': 'Dawud',       'سليمان': 'Sulaiman',
    'ادريس': 'Idris',      'عمرو': 'Amru',
    'عمر': 'Umar',         'زيد': 'Zaid',
    'اسامه': 'Usamah',     'اسامة': 'Usamah',
    'ايوب': 'Ayyub',       'أيوب': 'Ayyub',
    'طلحه': 'Thalhah',     'طلحة': 'Thalhah',
    'الزبير': 'Az-Zubair', 'زبير': 'Zubair',
    'سعد': "Sa'd",         'عمار': 'Ammar',
    'بلال': 'Bilal',       'جابر': 'Jabir',
    'اسحاق': 'Ishaq',      'إسحاق': 'Ishaq',
    'يزيد': 'Yazid',       'وكيع': 'Waki',
    'الليث': 'Al-Laits',   'ليث': 'Laits',
    'عطاء': 'Atha',        'مجاهد': 'Mujahid',
    'هلال': 'Hilal',       'صفوان': 'Shafwan',
    'سالم': 'Salim',       'نعيم': "Nu'aim",
    'الحسن': 'Al-Hasan',   'الحسين': 'Al-Husain',
    'اسماعيل': 'Ismail',   'إسماعيل': 'Ismail',
    'عبيد': 'Ubaid',       'عبيد الله': 'Ubaidullah',
    'مسلم': 'Muslim',
}

CHAR_MAP = {
    'ا': 'a', 'أ': 'a', 'إ': 'i', 'آ': 'aa',
    'ب': 'b', 'ت': 't', 'ث': 'ts', 'ج': 'j',
    'ح': 'h', 'خ': 'kh', 'د': 'd', 'ذ': 'dz',
    'ر': 'r', 'ز': 'z', 'س': 's', 'ش': 'sy',
    'ص': 'sh', 'ض': 'dh', 'ط': 'th', 'ظ': 'zh',
    'ع': "'", 'غ': 'gh', 'ف': 'f', 'ق': 'q',
    'ك': 'k', 'ل': 'l', 'م': 'm', 'ن': 'n',
    'ه': 'h', 'و': 'w', 'ي': 'y', 'ى': 'a',
    'ة': 'ah', 'ئ': 'i', 'ؤ': 'u', 'ء': "'", 'ـ': '',
    'َ': 'a', 'ِ': 'i', 'ُ': 'u',
    'ً': 'an', 'ٍ': 'in', 'ٌ': 'un', 'ْ': '', 'ّ': '',
}

def _transliterate_word(word):
    return ''.join(CHAR_MAP.get(c, '') if c in CHAR_MAP else c for c in word)

def arabic_to_indonesian(text):
    if not isinstance(text, str) or not text.strip():
        return text
    text_stripped = text.strip()
    if text_stripped in NAMA_KAMUS:
        return NAMA_KAMUS[text_stripped]
    words = text_stripped.split()
    result = []
    i = 0
    while i < len(words):
        if i + 1 < len(words):
            two = words[i] + ' ' + words[i + 1]
            if two in NAMA_KAMUS:
                result.append(NAMA_KAMUS[two])
                i += 2
                continue
        word = words[i]
        if word in NAMA_KAMUS:
            result.append(NAMA_KAMUS[word])
        elif word.startswith('ال'):
            core = word[2:]
            trans = NAMA_KAMUS.get(core, _transliterate_word(core))
            result.append('Al-' + trans.capitalize() if trans else word)
        else:
            trans = _transliterate_word(word)
            result.append(trans.capitalize() if trans else word)
        i += 1
    return ' '.join(result)

print("✅ Fungsi transliterasi siap digunakan.")

✅ Fungsi transliterasi siap digunakan.


In [36]:
edges_df['Nama_Guru']  = edges_df['Guru'].apply(arabic_to_indonesian)
edges_df['Nama_Murid'] = edges_df['Murid'].apply(arabic_to_indonesian)

edges_df = edges_df.drop(columns=['Id_Guru', 'Id_Murid', 'Id', 'Edge_id', 'Guru_Id', 'Murid_Id'], errors='ignore')
edges_df = edges_df.sort_values('Weight', ascending=False).reset_index(drop=True)
edges_df.insert(0, 'Edge_id', range(1, len(edges_df) + 1))

# Petakan Guru/Murid ke Perawi_ID dari narrator_df
perawi_map = narrator_df.set_index('Perawi')['Perawi_ID'].to_dict()

edges_df['Guru_Id']  = edges_df['Guru'].apply(normalize_arabic_name).map(perawi_map).astype('Int64')
edges_df['Murid_Id'] = edges_df['Murid'].apply(normalize_arabic_name).map(perawi_map).astype('Int64')

edges_df = edges_df[['Edge_id', 'Guru_Id', 'Guru', 'Nama_Guru', 'Murid_Id', 'Murid', 'Nama_Murid', 'Weight', 'Inverse_Weight']]

print("✅ Transliterasi selesai!")
print(f"Cek NaN Guru_Id : {edges_df['Guru_Id'].isna().sum()}")
print(f"Cek NaN Murid_Id: {edges_df['Murid_Id'].isna().sum()}")
display(edges_df.head(10))

✅ Transliterasi selesai!
Cek NaN Guru_Id : 0
Cek NaN Murid_Id: 1


,Edge_id,Guru_Id,Guru,Nama_Guru,Murid_Id,Murid,Nama_Murid,Weight,Inverse_Weight
0,1,4,ابن عمر,Ibnu Umar,13,نافع,Nafi',11346,0.000088
1,2,18,معمر,Ma'mar,21,عبد الرزاق,Abdurrazzaq,6402,0.000156
2,3,48,ابو هشام بن عروه,Abu Hisyam bin Urwah,33,هشام بن عروه,Hisyam bin Urwah,6390,0.000156
3,4,2,ابن عباس,Ibnu Abbas,41,عكرمه,Ikrimah,5456,0.000183
4,5,2,ابن عباس,Ibnu Abbas,51,سعيد بن جبير,Sa'id bin Jubair,4663,0.000214
5,6,1,ابو هريره,Abu Hurairah,34,ابو سلمه,Abu Salamah,4502,0.000222
6,7,7,الزهري,Az-Zuhri,18,معمر,Ma'mar,4381,0.000228
7,8,1,ابو هريره,Abu Hurairah,50,ابو صالح,Abu Shalih,3972,0.000252
8,9,3,عائشه,Aisyah,57,عروه,Urwah,3953,0.000253
9,10,3,عائشه,Aisyah,48,ابو هشام بن عروه,Abu Hisyam bin Urwah,3950,0.000253


In [37]:
from difflib import get_close_matches

# Daftar semua nama perawi yang sudah dinormalisasi
perawi_list = narrator_df['Perawi'].tolist()

def fuzzy_match_perawi(nama, cutoff=0.7):
    norm = normalize_arabic_name(nama)
    matches = get_close_matches(norm, perawi_list, n=1, cutoff=cutoff)
    if matches:
        match = matches[0]
        pid = narrator_df.loc[narrator_df['Perawi'] == match, 'Perawi_ID'].values[0]
        return pid, match
    return None, None

# ── Perbaiki NaN Guru_Id ────────────────────────────────────────────────────
nan_guru_mask = edges_df['Guru_Id'].isna()
if nan_guru_mask.sum() > 0:
    print(f"NaN Guru_Id sebelum: {nan_guru_mask.sum()}")
    for idx in edges_df[nan_guru_mask].index:
        pid, matched = fuzzy_match_perawi(edges_df.at[idx, 'Guru'])
        if pid:
            edges_df.at[idx, 'Guru_Id'] = pid
            print(f"  ✅ '{edges_df.at[idx, 'Guru']}' → '{matched}' (ID {pid})")
        else:
            print(f"  ❌ '{edges_df.at[idx, 'Guru']}' tidak cocok (tetap NaN)")
else:
    print("✅ Tidak ada NaN Guru_Id")

# ── Perbaiki NaN Murid_Id ───────────────────────────────────────────────────
nan_murid_mask = edges_df['Murid_Id'].isna()
if nan_murid_mask.sum() > 0:
    print(f"\nNaN Murid_Id sebelum: {nan_murid_mask.sum()}")
    for idx in edges_df[nan_murid_mask].index:
        pid, matched = fuzzy_match_perawi(edges_df.at[idx, 'Murid'])
        if pid:
            edges_df.at[idx, 'Murid_Id'] = pid
            print(f"  ✅ '{edges_df.at[idx, 'Murid']}' → '{matched}' (ID {pid})")
        else:
            print(f"  ❌ '{edges_df.at[idx, 'Murid']}' tidak cocok (tetap NaN)")
else:
    print("✅ Tidak ada NaN Murid_Id")

print(f"\nNaN Guru_Id  akhir : {edges_df['Guru_Id'].isna().sum()}")
print(f"NaN Murid_Id akhir : {edges_df['Murid_Id'].isna().sum()}")

sisa_nan = edges_df[edges_df['Guru_Id'].isna() | edges_df['Murid_Id'].isna()]
if len(sisa_nan):
    print(f"\nBaris yang masih NaN ({len(sisa_nan)} baris):")
    display(sisa_nan[['Guru', 'Murid', 'Weight']])

✅ Tidak ada NaN Guru_Id

NaN Murid_Id sebelum: 1
  ❌ 'عبده معلومات الرواه جويبر' tidak cocok (tetap NaN)

NaN Guru_Id  akhir : 0
NaN Murid_Id akhir : 1

Baris yang masih NaN (1 baris):


,Guru,Murid,Weight
718731,الضحاك,عبده معلومات الرواه جويبر,1


In [38]:
display(edges_df)

,Edge_id,Guru_Id,Guru,Nama_Guru,Murid_Id,Murid,Nama_Murid,Weight,Inverse_Weight
0,1,4,ابن عمر,Ibnu Umar,13,نافع,Nafi',11346,0.000088
1,2,18,معمر,Ma'mar,21,عبد الرزاق,Abdurrazzaq,6402,0.000156
2,3,48,ابو هشام بن عروه,Abu Hisyam bin Urwah,33,هشام بن عروه,Hisyam bin Urwah,6390,0.000156
3,4,2,ابن عباس,Ibnu Abbas,41,عكرمه,Ikrimah,5456,0.000183
4,5,2,ابن عباس,Ibnu Abbas,51,سعيد بن جبير,Sa'id bin Jubair,4663,0.000214
...,...,...,...,...,...,...,...,...,...
776793,776794,7379,كيسان,Kysan,2764,محمد بن ربيعه,Muhammad bin Rby'h,1,1.000000
776794,776795,115,عليا,'lya,11903,صفوان بن عيسي الزهري,Shafwan bin Isa Az-Zuhri,1,1.000000
776795,776796,70179,ناسا من الصحابه,Nasa Mn Al-Shhabh,448,جابر بن زيد,Jabir bin Zaid,1,1.000000
776796,776797,7,الزهري,Az-Zuhri,2927,عبد الرزاق بن همام,Abdurrazzaq bin Hmam,1,1.000000


In [39]:
display(narrator_df.head(10))

,Perawi_ID,Perawi,Jumlah
0,1,ابو هريره,51669
1,2,ابن عباس,48300
2,3,عائشه,33145
3,4,ابن عمر,28892
4,5,شعبه,26570
5,6,سفيان,26545
6,7,الزهري,22195
7,8,الاعمش,18182
8,9,انس بن مالك,17264
9,10,قتاده,17016


In [40]:
edges_df.to_csv("../data/processed/edge.csv", index=False, encoding='utf-8-sig')
print(f"✅ edge.csv disimpan — {len(edges_df):,} baris, kolom: {list(edges_df.columns)}")

✅ edge.csv disimpan — 776,798 baris, kolom: ['Edge_id', 'Guru_Id', 'Guru', 'Nama_Guru', 'Murid_Id', 'Murid', 'Nama_Murid', 'Weight', 'Inverse_Weight']
